# MWE of distributions

In [13]:
import qmcpy as qp 
import scipy as sp
import numpy as np
from scipy import stats

## True Measure Way
It seems that we can only specify one kind of independent marginal

In [14]:
unif_sampler = qp.Sobol(1)
Gauss_sampler = qp.Gaussian(unif_sampler)
Gauss_sampler(8)

array([[ 1.45705433],
       [-0.08955636],
       [ 0.3913917 ],
       [-0.70823077],
       [ 0.96621994],
       [-0.50047446],
       [ 0.30664763],
       [-2.03191135]])

## SciPy Wrapper Way
Here we can stack different independent marginals, which we cannot do with `truemeasure` built-in

In [15]:
unif_sampler = qp.Sobol(2)

In [16]:
indep_marginals = [stats.uniform(-1.0, 4.0), stats.norm(0.0, 1.0)]
uniform_plus_Gauss_sampler = qp.SciPyWrapper(unif_sampler, indep_marginals)
uniform_plus_Gauss_sampler(8)

array([[-0.07148969, -1.07066321],
       [ 2.56735771,  0.21150014],
       [ 0.77779183,  1.46796684],
       [ 1.73269446, -0.35120035],
       [-0.77371048,  0.47594887],
       [ 2.27103597, -1.15500105],
       [ 0.07521985, -0.24228264],
       [ 1.43672343,  0.9845172 ]])

## ProductMeasure Way

`SciPyWrapper` can combine independent SciPy-style marginals in one list.  
`ProductMeasure` generalizes this idea to QMCPy true measures.

Instead of writing one custom joint distribution, we can build a product from smaller independent true measures. If the child dimensions are $(d_1, \ldots, d_k)$, then the total dimension is

```math
d = \sum_{j=1}^{k} d_j.
```

A single d-dimensional QMC point is split into coordinate blocks. Each block is sent to its corresponding child true measure, and the transformed blocks are concatenated back together.

For example, if one child has dimension 1 and another child has dimension 1, then `ProductMeasure` uses a 2D sampler and returns samples with shape `(n, 2)`.

In [23]:

# ProductMeasure example: combine a QMCPy Uniform true measure with a QMCPy Gaussian true measure.

product_sampler = qp.Sobol(2, seed=7)

children = [
    qp.Uniform(
        qp.Sobol(1),
        lower_bound=-1.0,
        upper_bound=3.0,
    ),
    qp.Gaussian(
        qp.Sobol(1),
        mean=0.0,
        covariance=1.0,
    ),
]

uniform_plus_gauss_product = qp.ProductMeasure(
    sampler=product_sampler,
    children=children,
)

x_product = uniform_plus_gauss_product(8)

print(x_product)
print("shape:", x_product.shape)

[[ 1.88649425  1.37191467]
 [-0.34617786 -0.17726898]
 [ 2.94705021 -1.82020578]
 [ 0.7182662   0.14783478]
 [ 1.27266625 -0.58474159]
 [-0.99858294  0.67771653]
 [ 2.33321935  0.37528243]
 [ 0.06586397 -1.10040013]]
shape: (8, 2)


Here, `ProductMeasure` uses one two-dimensional Sobol sampler. The first coordinate block is sent to the uniform child true measure, and the second coordinate block is sent to the Gaussian child true measure.

The child samplers are used only to define the child true measures and their dimensions. The actual product samples come from the single sampler passed directly to `ProductMeasure`.

The output has shape `(8, 2)` because we requested 8 samples and the product measure has two one-dimensional children.

The first column comes from the uniform true measure on `[-1, 3]`.  
The second column comes from the Gaussian true measure.

This shows the main benefit of `ProductMeasure`: different QMCPy true measures can be combined as independent blocks without writing one new joint transform.

## Comparison with SciPyWrapper


In [18]:
# The same simple example can also be written using SciPyWrapper
# because both components are one-dimensional SciPy-style marginals.

same_sampler = qp.Sobol(2, seed=7)

indep_marginals = [
    stats.uniform(loc=-1.0, scale=4.0),
    stats.norm(loc=0.0, scale=1.0),
]

uniform_plus_gauss_scipy = qp.SciPyWrapper(
    same_sampler,
    indep_marginals,
)

x_scipy = uniform_plus_gauss_scipy(8)

print(x_scipy)
print("shape:", x_scipy.shape)

print("ProductMeasure mean:", x_product.mean(axis=0))
print("SciPyWrapper mean:", x_scipy.mean(axis=0))

[[ 1.88649425  1.37191467]
 [-0.34617786 -0.17726898]
 [ 2.94705021 -1.82020578]
 [ 0.7182662   0.14783478]
 [ 1.27266625 -0.58474159]
 [-0.99858294  0.67771653]
 [ 2.33321935  0.37528243]
 [ 0.06586397 -1.10040013]]
shape: (8, 2)
ProductMeasure mean: [ 0.98484993 -0.13873351]
SciPyWrapper mean: [ 0.98484993 -0.13873351]


For simple one-dimensional marginals, `SciPyWrapper` and `ProductMeasure` can express similar ideas.

The difference is that `ProductMeasure` works at the true-measure level. This means a child can itself be multi-dimensional, such as a 2D Gaussian block, and the product measure can still combine it with another independent component.

## 2D Gaussian + Zero-inflated exponential

In [22]:
# ProductMeasure example with different child dimensions:
#   child 1: 2D Gaussian
#   child 2: 1D zero-inflated exponential

# Total dimension = 2 + 1 = 3.

product_sampler = qp.Sobol(3, seed=11)

children = [
    qp.Gaussian(
        qp.Sobol(2),
        mean=[0.0, 0.0],
        covariance=[[1.0, 0.6], [0.6, 1.0]],
    ),
    qp.ZeroInflatedExpUniform(
        qp.Sobol(1),
        p_zero=0.4,
        lam=1.5,
    ),
]

gaussian_plus_zero_inflated = qp.ProductMeasure(
    sampler=product_sampler,
    children=children,
)

x = gaussian_plus_zero_inflated(4096)

print("shape:", x.shape)
print("Gaussian block shape:", x[:, :2].shape)
print("Zero-inflated block shape:", x[:, 2:].shape)
print("Zero rate:", np.mean(x[:, 2] == 0.0))
print("Gaussian empirical correlation:", np.corrcoef(x[:, :2].T)[0, 1])

shape: (4096, 3)
Gaussian block shape: (4096, 2)
Zero-inflated block shape: (4096, 1)
Zero rate: 0.39990234375
Gaussian empirical correlation: 0.6000410310701878


This example is the case that `SciPyWrapper` alone does not naturally cover.

The first child contributes two coordinates from a 2D Gaussian.  
The second child contributes one coordinate from the zero-inflated exponential.

Therefore, the final output has shape `(4096, 3)`.

The first two columns form the Gaussian block. The third column is the zero-inflated exponential block. The zero rate should be close to `p_zero = 0.4`, so about 40% of the third-coordinate samples should be exactly zero.

This is the main reason for adding `ProductMeasure`: it allows independent components with different dimensions to be combined cleanly.

## Importance sampling
With `truemeasure` we can change the measure from uniform to something, but I do not see how we can chain variable transformations

In [20]:
integrand_vs_Gauss = qp.CustomFun(true_measure = Gauss_sampler, g = lambda x : x[:,0]**2)
stop_criterion = qp.CubQMCNetG(integrand_vs_Gauss, abs_tol=1e-6)
sol, data = stop_criterion.integrate()
print("sol =", sol)
print(data)

sol = 1.000000079471714
Data (Data)
    solution        1.000
    comb_bound_low  1.000
    comb_bound_high 1.000
    comb_bound_diff 1.61e-06
    comb_flags      1
    n_total         2^(23)
    n               2^(23)
    time_integrate  6.335
CubQMCNetG (AbstractStoppingCriterion)
    abs_tol         1.00e-06
    rel_tol         0
    n_init          2^(10)
    n_limit         2^(35)
CustomFun (AbstractIntegrand)
Gaussian (AbstractTrueMeasure)
    mean            0
    covariance      1
    decomp_type     PCA
DigitalNetB2 (AbstractLDDiscreteDistribution)
    d               1
    replications    1
    randomize       LMS DS
    gen_mats_source joe_kuo.6.21201.txt
    order           RADICAL INVERSE
    t               63
    alpha           1
    n_limit         2^(32)
    entropy         330256389939449950956208422361103894585


### We try different tolerances and we see that sample size increases roughly like 1/tolerance

In [21]:
abs_tol_vec = [1e-3, 1e-4, 1e-5, 1e-6, 1e-7]
n_total_vec = np.zeros(len(abs_tol_vec), dtype=int)
for ii, abs_tol in enumerate(abs_tol_vec):
    _,data = qp.CubQMCNetG(integrand_vs_Gauss, abs_tol=abs_tol).integrate()
    n_total_vec[ii] = data.n_total

print("abs_tol_vec =", abs_tol_vec)
print("n_total_vec =", n_total_vec)
print("n_increase = ", n_total_vec/n_total_vec[0])

abs_tol_vec = [0.001, 0.0001, 1e-05, 1e-06, 1e-07]
n_total_vec = [   16384    65536  1048576  8388608 33554432]
n_increase =  [1.000e+00 4.000e+00 6.400e+01 5.120e+02 2.048e+03]


### So we might want to try importance sampling, say using a Gaussian with a different variance or Pareto, but I do not know how without rewriting the integrand